In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from imblearn.over_sampling import SMOTE
# ML Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
import warnings
warnings.filterwarnings('ignore')

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# ==========================================================
# Load Dataset
# ==========================================================
df = pd.read_csv("Employee-Attrition.csv")
print("Initial Shape:", df.shape)
print(df.info())
print("Columns:", df.columns.tolist())
print(df.describe())

In [ ]:
#Drop unwanted Col
drop_cols = ['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeNumber']
df = df.drop(columns=drop_cols)

In [ ]:
#Target Variable
df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})

In [ ]:
#Target Distribution
print(df['Attrition'].value_counts())

In [ ]:
#identify Categorical and Numeric Columns
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

In [ ]:
print("Categorical Columns:\n", cat_cols)
print("Numeric Columns:\n", num_cols)

In [ ]:
# Nominal categorical variables (no order)
nominal_cols = ['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']

In [ ]:
# Nominal categorical variables (no order)
nominal_cols = ['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']

# Ordinal categorical variables (if any exist in your dataset)
# For simplicity, let's assume 'Education' and 'JobInvolvement' are ordinal-like (if present)
ordinal_cols = []
if 'Education' in df.columns:
    ordinal_cols.append('Education')
if 'JobSatisfaction' in df.columns:
    ordinal_cols.append('JobSatisfaction')

In [ ]:
# Label Encoding for Ordinal Columns
le = LabelEncoder()
for col in ordinal_cols:
    df[col] = le.fit_transform(df[col])

In [ ]:
#One-Hot Encoding for Nominal Data
df = pd.get_dummies(df, columns=nominal_cols, drop_first=True)

In [ ]:
#EDA
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (10,6)

df_attrition = df.copy()
df_performance = df.copy()

In [ ]:
#Attrition Overview
plt.figure(figsize=(16,8))

sns.countplot(
    x='Attrition',
    data=df_attrition,
    palette=['#27ae60','#e74c3c']
)

plt.title("Employee Attrition Overview")
plt.xlabel("Attrition (0 = Stay, 1 = Leave)")
plt.ylabel("Number of Employees")

plt.grid(axis='y', alpha=0.3)

plt.show()

In [ ]:
#Tenure Distribution by Attrition
plt.figure(figsize=(16,8))

sns.histplot(
    data=df_attrition,
    x='YearsAtCompany',
    hue='Attrition',
    bins=20,
    kde=True,
    palette=['#2ecc71','#e74c3c'],
    alpha=0.6
)

plt.title("Employee Tenure Distribution by Attrition")
plt.xlabel("Years at Company")
plt.ylabel("Employee Count")

plt.grid(alpha=0.3)

plt.show()

In [ ]:
#Job Level vs Attrition
plt.figure(figsize=(16,8))

sns.barplot(
    x='JobLevel',
    y='Attrition',
    data=df_attrition,
    palette='viridis'
)

plt.title("Attrition Rate by Job Level")
plt.xlabel("Job Level")
plt.ylabel("Attrition Rate")

plt.grid(axis='y', alpha=0.3)

plt.show()

In [ ]:

# Performance Rating Distribution
plt.figure(figsize=(16,8))

sns.countplot(
    x='PerformanceRating',
    data=df_performance,
    palette='magma'
)

plt.title("Employee Performance Rating Distribution")
plt.xlabel("Performance Rating")
plt.ylabel("Employee Count")

plt.grid(axis='y', alpha=0.3)

plt.show()


In [ ]:
#Experience Distribution
plt.figure(figsize=(16,8))

sns.histplot(
    df_performance['YearsAtCompany'],
    bins=25,
    kde=True,
    color='#3498db'
)

plt.title("Employee Experience Distribution")
plt.xlabel("Years at Company")
plt.ylabel("Frequency")

plt.grid(alpha=0.3)

plt.show()

In [ ]:
#Monthly Income vs Attrition
plt.figure(figsize=(16,8))

sns.boxplot(
    x='Attrition',
    y='MonthlyIncome',
    data=df_attrition,
    palette=['#1abc9c','#e74c3c']
)

plt.title("Monthly Income Comparison by Attrition")
plt.xlabel("Attrition")
plt.ylabel("Monthly Income")

plt.grid(alpha=0.3)

plt.show()

In [ ]:
#Work Life Balance vs Attrition
plt.figure(figsize=(16,8))

sns.countplot(
    x='WorkLifeBalance',
    hue='Attrition',
    data=df_attrition,
    palette='Set3'
)

plt.title("Work Life Balance vs Attrition")
plt.xlabel("Work Life Balance Level")
plt.ylabel("Employee Count")

plt.grid(axis='y', alpha=0.3)

plt.show()

In [ ]:
#Correlation Heatmap"
plt.figure(figsize=(16,8))

sns.heatmap(
    df.corr(),
    cmap='coolwarm',
    linewidths=0.5
)

plt.title("Feature Correlation Heatmap")

plt.show()

In [ ]:
def visualize_numeric_features(dataframe, main_title="Numeric Feature Analysis", columns_per_row=3):

    numeric_features = dataframe.select_dtypes(include=['int64','float64']).columns

    total_features = len(numeric_features)
    rows = (total_features + columns_per_row - 1) // columns_per_row

    plt.figure(figsize=(6*columns_per_row,4*rows))
    plt.suptitle(main_title, fontsize=18, fontweight='bold')

    for idx, feature in enumerate(numeric_features):

        plt.subplot(rows, columns_per_row, idx+1)

        sns.histplot(
            dataframe[feature],
            bins=25,
            kde=True,
            color='#3498db'
        )

        plt.title(feature)
        plt.grid(alpha=0.2)

    plt.tight_layout()
    plt.show()

In [ ]:
visualize_numeric_features(df, "Employee Attrition Dataset - Numeric Distributions")

In [ ]:
print(df.select_dtypes(exclude=np.number).columns)

In [ ]:
# ==========================================================
# Plot Categorical Feature Distributions
# ==========================================================

def plot_categorical_distributions(df, title="Dataset", cols_per_row=4):

    # select categorical columns
    cat_cols = df.select_dtypes(exclude=np.number).columns.tolist()

    # if no categorical columns
    if not cat_cols:
        print("⚠️ No categorical columns found!")
        return

    n_cols = cols_per_row
    n_rows = (len(cat_cols) + n_cols - 1) // n_cols

    plt.figure(figsize=(5*n_cols, 5*n_rows))

    plt.suptitle(
        f"{title} - Categorical Feature Distributions",
        fontsize=18,
        fontweight='bold',
        y=1.02
    )

    # plot each categorical feature
    for i, col in enumerate(cat_cols, 1):

        plt.subplot(n_rows, n_cols, i)

        sns.countplot(
            x=df[col],
            palette="viridis"     # added color palette
        )

        plt.title(col, fontsize=12)
        plt.xlabel("")
        plt.ylabel("Count")

        plt.xticks(rotation=45)

        plt.grid(axis='y', alpha=0.3)

    plt.tight_layout()

    plt.show()


# ==========================================================
# Run Function
# ==========================================================

plot_categorical_distributions(
    df,
    title="Attrition Dataset - Categorical Features"
)

In [ ]:
# select numeric columns
numeric_columns = df_performance.select_dtypes(include=['int64','float64']).columns

# reshape dataframe for plotting
df_melt = df_performance[numeric_columns].melt(var_name="Feature", value_name="Value")

# create distribution plots
g = sns.FacetGrid(df_melt, col="Feature", col_wrap=4, sharex=False, sharey=False)

g.map(sns.histplot, "Value", bins=30, color="steelblue", kde=True)

g.fig.suptitle("Performance Dataset - Numeric Feature Distributions", fontsize=18)

plt.tight_layout()

plt.show()

In [ ]:
plot_categorical_distributions(df_performance, title="Performance Dataset - Categorical Features")

In [ ]:
# ==========================================================
# Define Target & Selected Features
# ==========================================================
target_attr = 'Attrition'
features_attr = [
    'OverTime', 'MaritalStatus', 'DistanceFromHome',
    'JobRole', 'Department', 'TotalWorkingYears',
    'JobLevel', 'YearsInCurrentRole', 'MonthlyIncome',
    'Age', 'YearsWithCurrManager', 'StockOptionLevel',
    'YearsAtCompany', 'JobInvolvement'
]
target_perf = 'PerformanceRating'
features_perf = [
    'YearsInCurrentRole', 'YearsWithCurrManager', 'YearsSinceLastPromotion',
    'TotalWorkingYears', 'DistanceFromHome', 'RelationshipSatisfaction',
    'EnvironmentSatisfaction', 'JobInvolvement'
]

X_attr = df.drop(columns=[target_attr])
y_attr = df[target_attr]

X_perf = df[features_perf]
y_perf = df['PerformanceRating']


In [ ]:
# ==========================================================
# Feature Scaling
# ==========================================================
scaler_attr = StandardScaler()
X_scaled_attr = scaler_attr.fit_transform(X_attr)

scaler_perf = StandardScaler()
X_scaled_perf = scaler_perf.fit_transform(X_perf)

In [ ]:
# ==========================================================
# Handle Class Imbalance (SMOTE)
# ==========================================================
smote = SMOTE(random_state=42)
X_res_attr, y_res_attr = smote.fit_resample(X_scaled_attr, y_attr)

print("Before SMOTE:\n", y_attr.value_counts())
print("After SMOTE:\n", y_res_attr.value_counts())

X_res_perf, y_res_perf = smote.fit_resample(X_scaled_perf, y_perf)

print("Before SMOTE:\n", y_perf.value_counts())
print("After SMOTE:\n", y_res_perf.value_counts())


In [ ]:
# ==========================================================
# Train-Test Split
# ==========================================================
X_train_attr, X_test_attr, y_train_attr, y_test_attr = train_test_split(
    X_res_attr, y_res_attr, test_size=0.2, random_state=42, stratify=y_res_attr
)
print("Train Shape:", X_train_attr.shape)
print("Test Shape:", X_test_attr.shape)

X_train_perf, X_test_perf, y_train_perf, y_test_perf = train_test_split(
    X_res_perf, y_res_perf, test_size=0.2, random_state=42, stratify=y_res_perf
)
print("Train Shape:", X_train_perf.shape)
print("Test Shape:", X_test_perf.shape)

In [ ]:
# ==========================================================
# Function: Train and Evaluate Multiple Models
# ==========================================================

def train_and_evaluate_models(X_train, y_train, X_test, y_test,target_name):
    """
    Train and evaluate multiple ML models on the given dataset.
    Returns a dictionary with accuracy, AUROC, and classification reports.
    """
    print("\n==============================")
    print(f"🚀 Training and Evaluating Models for: {target_name}")
    print("==============================")

    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
        "Random Forest": RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42),
        "Decision Tree": DecisionTreeClassifier(class_weight='balanced', random_state=42),
        "Naive Bayes": GaussianNB(),
        "SVM": SVC(kernel='rbf', class_weight='balanced', probability=True, random_state=42),
        "KNN": KNeighborsClassifier(n_neighbors=5),
        "Gradient Boosting": GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, random_state=42)
    }

    results = {}
    
    for name, model in models.items():
        print(f"\n==============================")
        print(f"🧠 {name}")
        print("==============================")

        # Train model
        model.fit(X_train, y_train)

        # Predictions
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

        # Metrics
        acc = accuracy_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_prob) if y_prob is not None else None
        report = classification_report(y_test, y_pred)

        print(f"{name} Accuracy: {acc * 100:.2f}%")
        if auc is not None:
            print(f"{name} AUROC: {auc:.4f}")
        print("Classification Report:\n", report)

        results[name] = {
            "model": model,
            "accuracy": acc,
            "auc": auc,
            "report": report
        }

    print("\n==============================")
    print(" Model Training Completed")
    print("==============================")
    return results

# Train and evaluate for Attrition and Performance
results_attr = train_and_evaluate_models(X_train_attr, y_train_attr, X_test_attr, y_test_attr, target_name="Employee Attrition")



In [ ]:
results_perf = train_and_evaluate_models(X_train_perf, y_train_perf, X_test_perf, y_test_perf, target_name="Performance Rating")

In [ ]:
# ==========================================================
# ⚙️ Function: Hyperparameter Tuning for Multiple Models
# ==========================================================
def tune_models(X_train, y_train, X_test, y_test, target_name="Target"):
    print("\n==============================")
    print(f"🔧 Hyperparameter Tuning for {target_name} Dataset")
    print("==============================")

    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
        "Random Forest": RandomForestClassifier(class_weight='balanced', random_state=42),
        "Decision Tree": DecisionTreeClassifier(class_weight='balanced', random_state=42),
        "Naive Bayes": GaussianNB(),
        "SVM": SVC(probability=True, class_weight='balanced', random_state=42),
        "KNN": KNeighborsClassifier(),
        "Gradient Boosting": GradientBoostingClassifier(random_state=42)
    }

    param_grids = {
        "Logistic Regression": {"C": [0.01, 0.1, 1, 10], "solver": ["lbfgs", "liblinear"]},
        "Random Forest": {"n_estimators": [100, 200, 300], "max_depth": [4, 6, 8, None]},
        "Decision Tree": {"max_depth": [4, 6, 8, None]},
        "Naive Bayes": {},  # no hyperparameters to tune
        "SVM": {"C": [0.1, 1, 10], "kernel": ["linear", "rbf"]},
        "KNN": {"n_neighbors": [3, 5, 7, 9], "weights": ["uniform", "distance"]},
        "Gradient Boosting": {"n_estimators": [100, 200], "learning_rate": [0.01, 0.05, 0.1]}
    }

    best_models = {}

    # ==========================================================
    # 🔁 Iterate and Tune Each Model
    # ==========================================================
    for name, model in models.items():
        print(f"\n▶️ Tuning {name}...")

        params = param_grids.get(name, {})
        if not params:
            print(f"⚠️ {name} has no hyperparameters to tune. Using default model.")
            model.fit(X_train, y_train)
            best_model = model
        else:
            grid = GridSearchCV(
                estimator=model,
                param_grid=params,
                cv=5,
                scoring='roc_auc',
                n_jobs=-1,
                verbose=0
            )
            grid.fit(X_train, y_train)
            best_model = grid.best_estimator_
            print(f"✅ Best Parameters for {name}: {grid.best_params_}")

        best_models[name] = best_model

        # ================= Evaluation =================
        y_pred = best_model.predict(X_test)

        # Handle both probability and decision function models
        if hasattr(best_model, "predict_proba"):
            y_proba = best_model.predict_proba(X_test)[:, 1]
        else:
            y_proba = best_model.decision_function(X_test)

        print(f"📊 Accuracy: {accuracy_score(y_test, y_pred):.3f}")
        print(f"🎯 AUC: {roc_auc_score(y_test, y_proba):.3f}")
        print(classification_report(y_test, y_pred, zero_division=0))

    # ==========================================================
    # ✅ Summary
    # ==========================================================
    print("\n==============================")
    print(f"🏁 Hyperparameter Tuning Completed for {target_name}")
    print("==============================")

    return best_models


# ==========================================================
# 📊 Example Usage for Multiple Targets
# ==========================================================
best_models_attr = tune_models(X_train_attr, y_train_attr, X_test_attr, y_test_attr, target_name="Attrition")
best_models_perf = tune_models(X_train_perf, y_train_perf, X_test_perf, y_test_perf, target_name="Performance Rating")